# Data Analytics Using Python — Practice Additions

Additional exam-style questions (Q11–Q20) with solutions and a 20-question numerical practice set.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, mean_squared_error, r2_score, roc_auc_score, roc_curve, f1_score, precision_score, recall_score, silhouette_score)
from sklearn.pipeline import Pipeline
sns.set(style='whitegrid', context='notebook')
plt.style.use('fivethirtyeight')
print('Libraries ready.')


## Additional Exam-Style Questions (Q11–Q20) with Solutions

### Q11: GridSearchCV for KNN (Pipeline)
Tune `n_neighbors` in KNN using a pipeline with scaling; report best k and score.

In [ ]:
iris_gs = sns.load_dataset('iris')
X_gs = iris_gs.drop(columns=['species'])
y_gs = LabelEncoder().fit_transform(iris_gs['species'])
pipe_gs = Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier())])
param_grid = {'knn__n_neighbors': [3,5,7,9,11]}
gs = GridSearchCV(pipe_gs, param_grid, cv=5, scoring='accuracy')
gs.fit(X_gs, y_gs)
print('Best k:', gs.best_params_['knn__n_neighbors'], 'Best CV accuracy:', round(gs.best_score_,3))


Grid search evaluates multiple hyperparameters under cross-validation and selects the configuration that generalizes best while avoiding leakage by encapsulating scaling inside the pipeline.

### Q12: Confusion Matrix and Macro-F1 (Multiclass Logistic)
Compute confusion matrix and macro-F1 on a train/test split with logistic regression.

In [ ]:
iris_cm = sns.load_dataset('iris')
X_cm = iris_cm.drop(columns=['species'])
y_cm = LabelEncoder().fit_transform(iris_cm['species'])
Xtr_cm, Xte_cm, ytr_cm, yte_cm = train_test_split(X_cm, y_cm, test_size=0.2, random_state=42, stratify=y_cm)
pipe_cm = Pipeline([('scaler', StandardScaler()), ('logreg', LogisticRegression(max_iter=1000))])
pipe_cm.fit(Xtr_cm, ytr_cm)
yp_cm = pipe_cm.predict(Xte_cm)
print('Confusion matrix:
', confusion_matrix(yte_cm, yp_cm))
print('Macro-F1:', round(f1_score(yte_cm, yp_cm, average='macro'),3))


Macro-F1 treats all classes equally, which is important when classes have differing frequencies. The confusion matrix summarizes per-class errors.

### Q13: Bootstrap CI for a Mean
Use bootstrap resampling to estimate a 95% confidence interval for a sample mean.

In [ ]:
np.random.seed(123)
sample = np.random.normal(100, 15, size=200)
boot_means = []
for i in range(1000):
    boot = np.random.choice(sample, size=len(sample), replace=True)
    boot_means.append(boot.mean())
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
print('Bootstrap 95% CI:', round(ci_low,2), round(ci_high,2))


Bootstrap approximates the sampling distribution by resampling with replacement, yielding non-parametric confidence intervals without strict normality assumptions.

### Q14: Binary Metrics from Confusion Matrix
For setosa vs others, compute accuracy, precision, recall, and F1 using logistic regression.

In [ ]:
iris_bin = sns.load_dataset('iris')
X_bin = iris_bin.drop(columns=['species'])
y_bin = (iris_bin['species'] == 'setosa').astype(int)
Xtr_b, Xte_b, ytr_b, yte_b = train_test_split(X_bin, y_bin, test_size=0.2, random_state=42)
pipe_bin = Pipeline([('scaler', StandardScaler()), ('logreg', LogisticRegression(max_iter=1000))])
pipe_bin.fit(Xtr_b, ytr_b)
yp_b = pipe_bin.predict(Xte_b)
print('Accuracy:', round(accuracy_score(yte_b, yp_b),3))
print('Precision:', round(precision_score(yte_b, yp_b),3))
print('Recall:', round(recall_score(yte_b, yp_b),3))
print('F1:', round(f1_score(yte_b, yp_b),3))


Accuracy summarizes overall correctness, while precision and recall capture positive-class trade-offs; F1 balances both in a single score.

### Q15: Coefficient Magnitudes as Feature Importance (Logistic)
Train logistic regression on standardized iris features and rank features by coefficient magnitude.

In [ ]:
iris_imp = sns.load_dataset('iris')
X_imp = iris_imp.drop(columns=['species'])
y_imp = LabelEncoder().fit_transform(iris_imp['species'])
sc = StandardScaler()
X_imp_s = sc.fit_transform(X_imp)
logc = LogisticRegression(max_iter=1000, multi_class='auto')
logc.fit(X_imp_s, y_imp)
coef_mag = np.mean(np.abs(logc.coef_), axis=0)
feat_order = np.array(X_imp.columns)[np.argsort(-coef_mag)]
print('Feature ranking:', list(feat_order))


With standardized inputs, larger coefficient magnitudes often indicate more discriminative features, though multicollinearity and interactions can affect interpretation.

### Q16: PCA Cumulative Explained Variance
Plot cumulative explained variance and report number of components to reach 95%.

In [ ]:
X_pca_ev = StandardScaler().fit_transform(sns.load_dataset('iris').drop(columns=['species']))
pca_ev = PCA().fit(X_pca_ev)
cum = np.cumsum(pca_ev.explained_variance_ratio_)
needed = int(np.argmax(cum >= 0.95) + 1)
plt.figure(figsize=(6,4)); plt.plot(np.arange(1, len(cum)+1), cum, marker='o'); plt.axhline(0.95, color='red', linestyle='--'); plt.title('Cumulative Explained Variance'); plt.xlabel('Components'); plt.ylabel('Cumulative Variance'); plt.show()
print('Components needed for ≥95%:', needed)


Cumulative variance indicates how many components retain the majority of information, guiding dimensionality reduction decisions.

### Q17: Silhouette Score for K-Means
Compute silhouette scores for k=2..6 on standardized iris features.

In [ ]:
X_sil = StandardScaler().fit_transform(sns.load_dataset('iris').drop(columns=['species']))
for k in range(2,7):
    km = KMeans(n_clusters=k, random_state=0)
    labels = km.fit_predict(X_sil)
    score = silhouette_score(X_sil, labels)
    print(k, round(score,3))


Higher silhouette scores indicate better-defined clusters; comparing k helps choose a compact and separated partitioning.

### Q18: Rolling Volatility
Compute a 14-day rolling standard deviation and compare with the rolling mean for volatility analysis.

In [ ]:
tsv_dates = pd.date_range('2024-03-01', periods=120, freq='D')
tsv_vals = pd.Series(np.random.randn(120).cumsum())
tsv = pd.DataFrame({'date': tsv_dates, 'value': tsv_vals}).set_index('date')
tsv['roll_mean'] = tsv['value'].rolling(14).mean()
tsv['roll_std'] = tsv['value'].rolling(14).std()
tsv[['value','roll_mean','roll_std']].plot(figsize=(10,4)); plt.title('Rolling Mean and Std'); plt.show()


Rolling standard deviation quantifies short-term variability; rising volatility with stable mean can indicate regime changes or noise spikes.

### Q19: Z-Score Outlier Flagging
Flag rows with |z| > 2 for numeric columns and count outliers.

In [ ]:
df_demo = pd.DataFrame({'x': np.concatenate([np.random.normal(0,1,100), [5,6]]), 'y': np.concatenate([np.random.normal(0,1,100), [-5]])})
z = np.abs(stats.zscore(df_demo))
flags = (z > 2).any(axis=1)
print('Outliers count:', int(flags.sum()))


Z-score thresholding is a simple outlier rule-of-thumb under approximate normality; flagged points may warrant further inspection.

### Q20: Linear Regression on Sales Data
Predict `Units` from `Price` and categories using one-hot encoding; evaluate CV R².

In [ ]:
sales_lr = pd.read_csv('practice_sales.csv')
sales_enc = pd.get_dummies(sales_lr, columns=['Region','Product'], drop_first=True)
X_sales = sales_enc.drop(columns=['Units','Date'])
y_sales = sales_enc['Units']
pipe_sales = Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())])
cv_sales = cross_val_score(pipe_sales, X_sales, y_sales, cv=5, scoring='r2')
print('CV R2 (mean ± std):', cv_sales.mean().round(3), '±', cv_sales.std().round(3))


Encoding categorical factors and scaling numeric features allow linear models to handle mixed data; CV R² assesses generalization of the predictive relationship.

## Numerical Questions — Practice Set (20)
Answer the following numerical problems. Show calculations and final results.

1) Given data [12, 15, 14, 10, 9, 16, 18], compute mean, median, variance, standard deviation, and IQR.
2) Two samples: A (mean=50, sd=8, n=40), B (mean=54, sd=10, n=35). Compute 95% CI for (B−A) using normal approximation.
3) From 100 trials, successes=45. Compute 95% CI for the proportion using normal approximation.
4) For pairs (x,y) = {(1,2),(2,3),(3,5),(4,4),(5,7)}, compute Pearson correlation r.
5) Compute z-score of value x=72 given population mean=60 and sd=9. Interpret.
6) Min-max scaling: scale x=37 to [0,1] given min=10, max=50. Also compute standardized value given mean=30, sd=8.
7) KNN distances from query q=(2,2) to points p1=(1,1), p2=(2,4), p3=(3,3), p4=(0,2). For k=3, what is the predicted class if classes are {A,B,A,B} respectively?
8) Given confusion matrix for binary class: TP=42, FP=8, FN=5, TN=45. Compute accuracy, precision, recall, and F1.
9) Linear regression on points {(1,2),(2,4),(3,5)}: compute slope and intercept via least squares.
10) PCA eigenvalues [3.2, 1.1, 0.7, 0.5]. Compute explained variance ratios and cumulative variance for first two components.
11) K-Means inertia: points assigned to centroid c1=(0,0): {(1,0),(0,1)} and c2=(3,3): {(4,3),(3,4)}. Compute total inertia (sum of squared distances).
12) Given dataset mean=120 and sd=15, using Chebyshev’s inequality, bound proportion within k=2 sd.
13) For a time series values [10,12,13,15,14,16], compute 3-day rolling mean at index of value 15.
14) Group means: Dept A scores [80,85,90], Dept B scores [70,75,78]. Compute pooled standard deviation and Cohen’s d effect size.
15) Logistic function: compute probability p for z=1.2 (p=1/(1+e^{-z})).
16) Odds and log-odds: if p=0.7, compute odds and log-odds.
17) Standard error of the mean: sample sd=12, n=64. Compute SEM and 95% CI for mean 100 (normal approximation).
18) Correlation to regression slope: given r=0.8, sd_x=5, sd_y=10, compute slope b1=r*(sd_y/sd_x).
19) Outlier detection by z-score: values [0,0.5,−0.2,3.5,−4.1,0.1]. Which values exceed |z|>2 assuming mean≈0 and sd≈1?
20) Weighted mean: weights w=[2,1,3], values v=[10,20,30]. Compute weighted average.
